# How look-alike wallet addresses are generated
**Educational demo** for the 0xUnstuck video *"How Scammers Generate Look-Alike Wallet Addresses"*.

> ⚠️ This grinds a vanity address for **yourself** (the legitimate use, like `0xC0FFEE...`).
> Generating one that mimics **someone else's** address to trick them into sending funds is **theft** — never do it.

The point: see **why the first/last characters are cheap to fake, and the middle never is** — so you know to check the middle.


In [ ]:
!pip install -q eth-account


In [ ]:
"""
Vanity address grinder — EDUCATIONAL DEMO
=========================================
Shows HOW a "look-alike" address is made: guess a key pair, derive the address,
check ONLY the characters a human actually looks at, throw it away, repeat.

⚠️  This grinds a vanity address for YOURSELF (the legitimate use, like 0xC0FFEE...).
    Making one that mimics SOMEONE ELSE'S address to trick them into sending funds is
    theft — never do it. The point is to see WHY the ends are cheap to fake and the
    middle never can be, so you know to check the middle.

Google Colab:  !pip install eth-account    then paste this file and run.
"""
import time
from eth_account import Account

def grind(prefix: str, budget_s: float = 30.0):
    target, tried, t0 = prefix.lower(), 0, time.time()
    while time.time() - t0 < budget_s:
        addr = Account.create().address          # 1+2. random key -> address
        tried += 1
        if addr[2:2 + len(target)].lower() == target:   # 3. compare ONLY the start
            return addr, tried, time.time() - t0
        # 4. no match -> discard, loop
    return None, tried, time.time() - t0

if __name__ == "__main__":
    print("Each extra hex character = 16x more work.\n")
    rate = None
    for p in ("a", "ab", "abc"):
        addr, tried, secs = grind(p, 30)
        rate = tried / secs
        if addr:
            print(f"  0x{p:<4} FOUND  {addr}")
            print(f"        {tried:>8,} tries · {secs:5.2f}s")
        else:
            print(f"  0x{p:<4} not found in {secs:.0f}s ({tried:,} tries)")
    print(f"\n  this laptop, pure Python: ~{rate:,.0f} addresses/sec\n")
    print("  extrapolating the SAME loop:")
    for n in (4, 6, 8, 10, 40):
        need = 16 ** n
        secs = need / rate
        if   secs < 60:      human = f"{secs:.0f} seconds"
        elif secs < 3600:    human = f"{secs/60:.0f} minutes"
        elif secs < 86400:   human = f"{secs/3600:.0f} hours"
        elif secs < 3.15e7:  human = f"{secs/86400:.0f} days"
        else:                human = f"{secs/3.15e7:.3g} years"
        label = f"{n} chars" + (" (the WHOLE address)" if n == 40 else "")
        print(f"    {label:<28} ~{need:.2e} tries   ->  {human}")
    print("\n  A GPU rig does this ~millions of times faster — which moves the")
    print("  first few / last few characters into 'seconds', and moves the FULL")
    print("  address absolutely nowhere. That's why the middle is always genuine.")


---
## Now on a GPU

Set **Runtime → Change runtime type → T4 GPU**, then run the cell below.


In [ ]:
# ============================================================
#  Why a GPU changes the economics  (run with Runtime > Change runtime type > T4 GPU)
# ============================================================
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

# The search is "embarrassingly parallel": every guess is independent of every other
# one, so thousands of GPU cores can test candidates simultaneously.
#
# HONEST SCOPE — read this before quoting the number:
#   This times the SEARCH + MATCH step at GPU scale. A production grinder also runs
#   the secp256k1 key->address derivation on the GPU, and that step dominates, which
#   is why real GPU grinders land around 10^8 addresses/sec, not the raw figure here.
#   This is deliberately NOT a fast grinder — the point is the parallelism, and the
#   fact that even at these speeds the FULL address stays permanently out of reach.

from numba import cuda
import numpy as np, time

@cuda.jit
def match_prefix(cands, want, out):
    i = cuda.grid(1)
    if i < cands.shape[0]:
        ok = 1
        for k in range(want.shape[0]):
            if cands[i, k] != want[k]:
                ok = 0
                break
        out[i] = ok

N = 40_000_000            # candidates tested in one shot
PREFIX = np.array([10, 11, 12], dtype=np.uint8)   # "abc" as hex nibbles

cands = np.random.randint(0, 16, size=(N, PREFIX.size), dtype=np.uint8)
d_c, d_w = cuda.to_device(cands), cuda.to_device(PREFIX)
d_o = cuda.device_array(N, dtype=np.uint8)

threads = 256
blocks = (N + threads - 1) // threads
match_prefix[blocks, threads](d_c, d_w, d_o)   # warm-up / JIT
cuda.synchronize()

t0 = time.time()
match_prefix[blocks, threads](d_c, d_w, d_o)
cuda.synchronize()
gpu_s = time.time() - t0

hits = int(d_o.copy_to_host().sum())
rate = N / gpu_s
print(f"\nGPU tested {N:,} candidates in {gpu_s*1000:.1f} ms  ->  {rate:,.0f} checks/sec")
print(f"matches found: {hits:,}  (expected ~{N/16**PREFIX.size:,.0f})")

print("\nAt that rate, matching N characters would take:")
for n in (4, 6, 8, 10, 12, 40):
    secs = 16**n / rate
    if   secs < 1:      human = f"{secs*1000:.1f} milliseconds"
    elif secs < 60:     human = f"{secs:.1f} seconds"
    elif secs < 3600:   human = f"{secs/60:.1f} minutes"
    elif secs < 86400:  human = f"{secs/3600:.1f} hours"
    elif secs < 3.15e7: human = f"{secs/86400:.1f} days"
    else:               human = f"{secs/3.15e7:.3g} years"
    tag = "  <-- the WHOLE address" if n == 40 else ""
    print(f"   {n:>2} chars   {16**n:.2e} guesses   ->  {human}{tag}")

print("\nEven with the search step running this fast, the full 40-character address")
print("is unreachable. The ends are cheap. The middle is not for sale.")
